### O JOIN no SQL
- Agora nosso banco de dados (BaseDados2) possui 3 tabelas:
    - alunos: Dados dos alunos da Hashtag
    - plataforma: Informações de acesso a plataforma
    - provas: Provas liberadas para o aluno e nota das provas feitas
- Base disponível em:
    - https://drive.google.com/file/d/11uIXRsHnC84mhx344CR1hQxqPQ43j4B0/view?usp=sharing

In [62]:
# Importando o pandas
import pandas as pd

In [63]:
# Criando a conexão
import sqlite3
con = sqlite3.connect('BaseDados2.db')

# Carregando as abas do Excel para tabelas no SQLite
abas_tabelas = {
    'Alunos': 'alunos',
    'Plataforma': 'plataforma',
    'Provas': 'provas'
}

for aba, tabela in abas_tabelas.items():
    base = pd.read_excel('BaseDados2.xlsx', sheet_name=aba)
    base.to_sql(tabela, con, index=False, if_exists='replace')

In [64]:
# Criando um cursor
cur = con.cursor()

In [65]:
# Criando uma função para consultar os dados
def executa_sql(comando):
    cur.execute(comando)
    resultado = cur.fetchall()
    resultado = pd.DataFrame(resultado)
    if resultado.shape[1] > 0:
        resultado.columns = [i[0] for i in cur.description]
    print(resultado.shape)
    display(resultado.head())
    return resultado

In [66]:
# Visualizando a tabela alunos
alunos = executa_sql('SELECT * FROM alunos')

(20, 4)


,id_aluno,nome_aluno,cod_matricula,e-mail
0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com
1,2,Bárbara da Cunha,63546,barbara@hashtag.com
2,3,Kevin Melo,80515,kevin@hashtag.com
3,4,Pedro Henrique da Costa,68004,pedro@hashtag.com
4,5,Mirella Viana,28421,mirella@hashtag.com


In [67]:
# Visualizando a tabela plataforma
plataforma = executa_sql('SELECT * FROM plataforma')

(21, 5)


,id_plataforma,cod_matricula,acesso_plataforma,acesso_liberado,dias_ultimo_acesso
0,1,38273,0,1,12
1,2,63546,0,1,2
2,3,80515,1,1,3
3,4,68004,1,1,1
4,5,28421,1,1,1


In [68]:
# Visualizando a tabela provas
provas = executa_sql('SELECT * FROM provas')

(43, 5)


,id_prova,cod_matricula,nr_prova,prova_feita,nota_prova
0,1,38273,1,1,9.0
1,2,63546,1,1,10.0
2,3,80515,1,1,7.0
3,4,68004,1,1,4.0
4,5,28421,1,1,7.0


### O merge do Pandas
- No Projeto 2, falamos do merge no pandas. Relembrando:

*O <font color='blue'>.merge()</font> irá juntar duas bases*
- Para isso, devemos passar:
    - base 1
    - base 2
    - how: forma que iremos fazer essa junção das bases
        - inner: o que tiver em comum entre as 2 bases (base 1 E base 2)
        - outer: tudo o que tiver nas 2 bases (base 1 OU base 2)
        - left: tudo o que tem na PRIMEIRA base, juntando com o que tiver na segunda
        - right: tudo o que tem na SEGUNDA base, juntando com o que tiver na primeira
    - on: colunar que vamos usar para fazer a junção da base

In [69]:
# Criando 2 dataframes
dic1 = {
    "nomes": ['Nome1','Nome2','Nome3'],
    "valores": [1,2,3]
}

base_dic1 = pd.DataFrame(dic1)

dic2 = {
    "nomes": ['Nome1','Nome2','Nome4'],
    "valores": [9,8,7]
}

base_dic2 = pd.DataFrame(dic2)

In [71]:
base_dic2

,nomes,valores
0,Nome1,9
1,Nome2,8
2,Nome4,7


In [75]:
base_merge = pd.merge(
    base_dic1, # <- primeira base
    base_dic2, # <- segunda base
    how='outer', # <- tipo de junção que vamos fazer
    on="nomes" # <- coluna que vamos usar para fazer essa junção das bases
) 

display(base_merge)

,nomes,valores_x,valores_y
0,Nome1,1.0,9.0
1,Nome2,2.0,8.0
2,Nome3,3.0,NaN
3,Nome4,NaN,7.0


In [79]:
# Unindo a tabela alunos com a tabela plataforma
base_merge = pd.merge(
    alunos, # <- primeira base
    plataforma, # <- segunda base
    how='right', # <- tipo de junção que vamos fazer
    on="cod_matricula" # <- coluna que vamos usar para fazer essa junção das bases
) 

display(base_merge)

,id_aluno,nome_aluno,cod_matricula,e-mail,id_plataforma,acesso_plataforma,acesso_liberado,dias_ultimo_acesso
0,1.0,Maria Eduarda da Rocha,38273,maria@hashtag.com,1,0,1,12
1,2.0,Bárbara da Cunha,63546,barbara@hashtag.com,2,0,1,2
2,3.0,Kevin Melo,80515,kevin@hashtag.com,3,1,1,3
3,4.0,Pedro Henrique da Costa,68004,pedro@hashtag.com,4,1,1,1
4,5.0,Mirella Viana,28421,mirella@hashtag.com,5,1,1,1
5,6.0,Lívia Jesus,22284,lívia@hashtag.com,6,0,1,14
6,7.0,Lara Lopes,38362,lara@hashtag.com,7,1,1,1
7,8.0,Lucca Cardoso,45109,lucca@hashtag.com,8,1,1,3
8,9.0,Isabelly Souza,31859,isabelly@hashtag.com,9,1,1,3
9,10.0,Cauã Porto,42674,cauã@hashtag.com,10,0,1,2


In [78]:
# Verificando se algum aluno não possui registro na tabela plataforma
base_merge[base_merge.id_plataforma.isnull()]

,id_aluno,nome_aluno,cod_matricula,e-mail,id_plataforma,acesso_plataforma,acesso_liberado,dias_ultimo_acesso


In [80]:
# Verificando se algum registro da tabela plataforma não possui a matrícula na tabela aluno
base_merge[base_merge.id_aluno.isnull()]

,id_aluno,nome_aluno,cod_matricula,e-mail,id_plataforma,acesso_plataforma,acesso_liberado,dias_ultimo_acesso
20,NaN,NaN,91047,NaN,21,0,0,0


In [82]:
alunos[alunos.cod_matricula == 91047]

,id_aluno,nome_aluno,cod_matricula,e-mail


In [83]:
base_merge.shape

(21, 8)